# Inhibitor and read arcs

Two extensions to ordinary arcs:

- **Inhibitor arc** (`--o`): the transition is enabled only if the place has *fewer* than `weight` tokens. It can bound a counter or test for emptiness.
- **Read arc** (dotted): needs tokens in the place but does *not* consume them.

```mermaid
flowchart LR
    put[put] --> Buf((Buf))
    Buf --o put
    Buf --> get[get]
```

In [1]:
from petrilab.petri import analysis, examples
from petrilab.petri.snakes_bridge import explore, to_snakes

n = examples.bounded_counter()
space = explore(to_snakes(n), token_cap=10)
print("bounded:", analysis.is_bounded(space))
print("max tokens in Buf:", analysis.max_tokens(space, "Buf"))
print("states:", len(space))
print("deadlock free:", analysis.is_deadlock_free(space))
print("live:", analysis.is_live(space, set(n.transitions)))

bounded: True
max tokens in Buf: 2
states: 3
deadlock free: True
live: True


The inhibitor arc on `Buf` with weight 2 disables `put` as soon as `Buf` holds 2 tokens. The net is therefore bounded with 3 states (0, 1, 2 tokens), deadlock-free and live.

In [2]:
space = explore(to_snakes(examples.bounded_counter(inhibited=False)), token_cap=10)
print("bounded:", analysis.is_bounded(space))

bounded: False


Without the inhibitor, `put` is always enabled, tokens exceed the cap and the net is reported unbounded.

In [3]:
n = examples.read_arc_net()
g = n.reachable()
for m in g:
    print(n.marking_dict(m))
print("Key always 1:", all(m[0] == 1 for m in g))

{'Key': 1, 'Job': 2, 'Done': 0}
{'Key': 1, 'Job': 1, 'Done': 1}
{'Key': 1, 'Job': 0, 'Done': 2}
Key always 1: True


`use` reads `Key` (needs 1 token) but never consumes it, so two jobs are processed using the same key. `Key` stays at 1 in every marking.

In [4]:
n = examples.read_arc_net(key=False)
print(n.reachable() == {n.initial_marking: []})

True


Without the key the transition is never enabled, and the initial marking is already dead.